# AI-NIDS: Machine Learning Training Pipeline
## Leakage-Resistant, Calibrated Hybrid Intrusion Detection System

This notebook contains the complete, reproducible training pipeline for the AI-NIDS intrusion detection system.

### Key Improvements & Audit Fixes Applied:
1. **Canonical 14-Feature Schema:** Eliminated the non-existent `Protocol` column; strictly aligns with genuine continuous flow features from CICIDS2017.
2. **Deduplication & Noise Removal:** Removed exact duplicate flows and resolved contradictory feature tuples prior to splitting.
3. **Preprocessing Isolation:** `StandardScaler` is fitted strictly on the training partition (`X_train`) to eliminate data leakage.
4. **Corrected Isolation Forest:** Trained exclusively on clean **benign flows** (`y_train == 0`) with score calibration, resolving previous anomaly inversion.
5. **Independent Multi-Model Evaluation:** Rigorous holdout test evaluation ($N = 64,191$) reporting separate metrics for Random Forest alone, Isolation Forest alone, and the Hybrid Ensemble ($S = 0.6 P_{RF} + 0.4 S_{IF}$).


### Cell 1: Environment & Library Imports
Imports standard scientific computing, data processing, and scikit-learn machine learning libraries.


In [1]:
#!/usr/bin/env python3
import os
import json
import joblib
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_curve
)

# Optional SHAP explainability
try:
    import shap
    print('[INFO] SHAP library found.')
except ImportError:
    shap = None
    print('[INFO] SHAP library not found, skipping SHAP-related features.')

print('All libraries imported successfully.')


[INFO] SHAP library not found, skipping SHAP-related features.
All libraries imported successfully.


### Cell 2: Configuration & Canonical 14-Feature Schema
Defines the canonical 14 flow features genuinely present in the benchmark dataset and specifies path locations.


In [2]:
# ===============================================================
# CONFIGURATION & FEATURE DEFINITIONS
# ===============================================================
csv_path = os.path.join('data', 'CICIDS2017_subset.csv')
if not os.path.exists(csv_path):
    # Fallback to root if run from a different subfolder
    csv_path = 'CICIDS2017_subset.csv' if os.path.exists('CICIDS2017_subset.csv') else csv_path

# Canonical 14-feature schema (Protocol removed as it is absent from raw dataset)
FEATURE_KEYS = [
    'Flow Duration',
    'Total Fwd Packets',
    'Total Backward Packets',
    'Total Length of Fwd Packets',
    'Total Length of Bwd Packets',
    'Fwd Packet Length Mean',
    'Bwd Packet Length Mean',
    'Flow IAT Mean',
    'Fwd IAT Mean',
    'Bwd IAT Mean',
    'Fwd PSH Flags',
    'Bwd PSH Flags',
    'Fwd URG Flags',
    'Bwd URG Flags'
]

LABEL_COL = 'Label'

print(f'Dataset path: {csv_path}')
print(f'Canonical feature count: {len(FEATURE_KEYS)}')
for i, f in enumerate(FEATURE_KEYS, 1):
    print(f'  {i:2d}. {f}')


Dataset path: data\CICIDS2017_subset.csv
Canonical feature count: 14
   1. Flow Duration
   2. Total Fwd Packets
   3. Total Backward Packets
   4. Total Length of Fwd Packets
   5. Total Length of Bwd Packets
   6. Fwd Packet Length Mean
   7. Bwd Packet Length Mean
   8. Flow IAT Mean
   9. Fwd IAT Mean
  10. Bwd IAT Mean
  11. Fwd PSH Flags
  12. Bwd PSH Flags
  13. Fwd URG Flags
  14. Bwd URG Flags


### Cell 3: Data Loading, Cleaning & Leakage-Resistant Deduplication
Loads the raw dataset, sanitizes invalid values (infinities, NaN), removes exact duplicate records, and purges contradictory feature tuples with conflicting labels.


In [3]:
# ===============================================================
# LOAD, SANITIZE & DEDUPLICATE DATA
# ===============================================================
print(f'[INFO] Loading dataset: {csv_path}...')
df_raw = pd.read_csv(csv_path)
print(f'[INFO] Raw dataset shape: {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns')

# 1. Strip whitespace from column headers
df_raw.columns = df_raw.columns.str.strip()

# 2. Sanitize infinite and NaN values
df_raw.replace([np.inf, -np.inf], np.nan, inplace=True)
df_raw[FEATURE_KEYS] = df_raw[FEATURE_KEYS].fillna(0.0)

# 3. Map binary labels: Benign -> 0, Attack -> 1
y_raw = df_raw[LABEL_COL].apply(lambda x: 0 if str(x).strip().upper() in ('BENIGN', 'NORMAL', '0') else 1)
df_raw['target'] = y_raw

# 4. Deduplication: Remove exact duplicate rows across features + label
dups_count = df_raw.duplicated(subset=FEATURE_KEYS + ['target']).sum()
print(f'[INFO] Exact duplicate rows identified: {dups_count:,}')
df_dedup = df_raw.drop_duplicates(subset=FEATURE_KEYS + ['target']).copy()
print(f'[INFO] Rows after exact deduplication: {len(df_dedup):,}')

# 5. Remove contradictory rows (identical feature tuples with opposite labels)
conflict_mask = df_dedup.duplicated(subset=FEATURE_KEYS, keep=False)
conflict_count = conflict_mask.sum()
print(f'[INFO] Contradictory rows with conflicting labels: {conflict_count}')
df_clean = df_dedup[~conflict_mask].copy()

X = df_clean[FEATURE_KEYS].astype(np.float64)
y = df_clean['target'].values

print(f'[INFO] Clean dataset size: {len(df_clean):,} rows')
print(f'       - Benign (0): {np.sum(y == 0):,} ({np.sum(y == 0)/len(y)*100:.2f}%)')
print(f'       - Attack (1): {np.sum(y == 1):,} ({np.sum(y == 1)/len(y)*100:.2f}%)')


[INFO] Loading dataset: data\CICIDS2017_subset.csv...
[INFO] Raw dataset shape: 225,745 rows, 79 columns
[INFO] Exact duplicate rows identified: 11,733
[INFO] Rows after exact deduplication: 214,012
[INFO] Contradictory rows with conflicting labels: 42
[INFO] Clean dataset size: 213,970 rows
       - Benign (0): 86,365 (40.36%)
       - Attack (1): 127,605 (59.64%)


### Cell 4: Stratified Train / Validation / Test Partitioning
Partitions the clean data using a 70% Train+Val / 30% Holdout Test split with stratification by class label. Train+Val is further split into an 85% Train / 15% Validation set.


In [4]:
# ===============================================================
# STRATIFIED PARTITIONING
# ===============================================================
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.15, random_state=42, stratify=y_train_val
)

print(f'[INFO] Partitioning Summary:')
print(f'       Training set:   {len(y_train):,} samples (Benign: {np.sum(y_train==0):,}, Attack: {np.sum(y_train==1):,})')
print(f'       Validation set: {len(y_val):,} samples (Benign: {np.sum(y_val==0):,}, Attack: {np.sum(y_val==1):,})')
print(f'       Holdout Test:   {len(y_test):,} samples (Benign: {np.sum(y_test==0):,}, Attack: {np.sum(y_test==1):,})')


[INFO] Partitioning Summary:
       Training set:   127,312 samples (Benign: 51,387, Attack: 75,925)
       Validation set: 22,467 samples (Benign: 9,068, Attack: 13,399)
       Holdout Test:   64,191 samples (Benign: 25,910, Attack: 38,281)


### Cell 5: Preprocessing Isolation (Feature Scaling)
Fits the `StandardScaler` strictly on the training partition (`X_train`) to prevent test set data leakage. Then transforms `X_train`, `X_val`, and `X_test`.


In [5]:
# ===============================================================
# PREPROCESSING ISOLATION
# ===============================================================
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=FEATURE_KEYS)
X_val_scaled = pd.DataFrame(scaler.transform(X_val), columns=FEATURE_KEYS)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=FEATURE_KEYS)

print(f'[INFO] StandardScaler fitted strictly on X_train ({scaler.n_features_in_} features).')
print(f'[INFO] Scaler mean shape: {scaler.mean_.shape}, scale shape: {scaler.scale_.shape}')


[INFO] StandardScaler fitted strictly on X_train (14 features).
[INFO] Scaler mean shape: (14,), scale shape: (14,)


### Cell 6: Train Supervised Model (Random Forest)
Trains a 100-tree `RandomForestClassifier` on the scaled training features and target labels.


In [6]:
# ===============================================================
# MODEL 1: SUPERVISED RANDOM FOREST
# ===============================================================
print('[INFO] Training RandomForestClassifier (n_estimators=100, random_state=42)...')
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_scaled, y_train)
print('[INFO] RandomForestClassifier training complete.')


[INFO] Training RandomForestClassifier (n_estimators=100, random_state=42)...
[INFO] RandomForestClassifier training complete.


### Cell 7: Train Semi-Supervised Model (Isolation Forest)
Trains an `IsolationForest` strictly on **clean benign instances** (`y_train == 0`) from the training set. Calibrates anomaly scores using empirical percentiles from the benign training distribution.


In [7]:
# ===============================================================
# MODEL 2: CORRECTED SEMI-SUPERVISED ISOLATION FOREST
# ===============================================================
X_train_benign = X_train_scaled[y_train == 0]
print(f'[INFO] Fitting IsolationForest on {len(X_train_benign):,} clean BENIGN training instances...')

iso = IsolationForest(n_estimators=100, contamination=0.03, random_state=42, n_jobs=-1)
iso.fit(X_train_benign)
print('[INFO] IsolationForest training complete.')

# Calibration percentiles on benign baseline
benign_scores = iso.score_samples(X_train_benign)
calib_min = float(np.percentile(benign_scores, 1))
calib_max = float(np.percentile(benign_scores, 99))
print(f'[INFO] Benign training score calibration: min_1% = {calib_min:.4f}, max_99% = {calib_max:.4f}')

def normalize_iso_score(raw_scores):
    """Converts raw isolation score to a normalized [0, 1] threat index."""
    return np.clip((calib_max - raw_scores) / (calib_max - calib_min), 0.0, 1.0)


[INFO] Fitting IsolationForest on 51,387 clean BENIGN training instances...
[INFO] IsolationForest training complete.
[INFO] Benign training score calibration: min_1% = -0.6849, max_99% = -0.3229


### Cell 8: (Optional) Initialize SHAP Explainer
Initializes a `shap.TreeExplainer` on the trained Random Forest model for local feature attribution.


In [8]:
# ===============================================================
# SHAP EXPLAINER (Optional)
# ===============================================================
explainer = None
if shap is not None:
    try:
        print('[INFO] Initializing SHAP TreeExplainer on Random Forest...')
        explainer = shap.TreeExplainer(rf)
        print('[INFO] SHAP TreeExplainer initialized successfully.')
    except Exception as e:
        print(f'[WARN] Could not initialize SHAP explainer: {e}')
else:
    print('[INFO] SHAP library not available; skipping TreeExplainer.')


[INFO] SHAP library not available; skipping TreeExplainer.


### Cell 9: Comprehensive Holdout Test Evaluation
Evaluates the Random Forest model alone, the Isolation Forest model alone, and the full Hybrid Ensemble ($S = 0.6 P_{RF} + 0.4 S_{IF}$, $T=0.50$) on the untouched holdout test set ($N=64,191$).


In [9]:
# ===============================================================
# EVALUATION ON UNTOUCHED HOLDOUT TEST SET (N = 64,191)
# ===============================================================
print('[INFO] Running inference on untouched holdout test partition...')

# 1. Random Forest Predictions & Probabilities
test_rf_prob = rf.predict_proba(X_test_scaled)[:, 1]
test_rf_pred = (test_rf_prob >= 0.50).astype(int)

# 2. Isolation Forest Scores & Predictions
test_iso_raw = iso.score_samples(X_test_scaled)
test_iso_prob = normalize_iso_score(test_iso_raw)
test_iso_pred = (test_iso_prob >= 0.50).astype(int)

# 3. Hybrid Ensemble (alpha=0.6, beta=0.4, threshold=0.50)
test_ensemble_score = 0.6 * test_rf_prob + 0.4 * test_iso_prob
test_ensemble_pred = (test_ensemble_score >= 0.50).astype(int)

def evaluate_predictions(name, y_true, y_pred, y_prob):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    auc = roc_auc_score(y_true, y_prob)
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    
    print(f'\n================== {name.upper()} ==================')
    print(f'Accuracy:    {acc*100:.4f}% ({acc:.6f})')
    print(f'Precision:   {prec*100:.4f}% ({prec:.6f})')
    print(f'Recall:      {rec*100:.4f}% ({rec:.6f})')
    print(f'F1-Score:    {f1*100:.4f}% ({f1:.6f})')
    print(f'Specificity: {spec*100:.4f}%')
    print(f'ROC-AUC:     {auc:.6f}')
    print(f'Confusion Matrix: TN={tn:,}, FP={fp:,}, FN={fn:,}, TP={tp:,}')
    return {
        'accuracy': float(acc), 'precision': float(prec), 'recall': float(rec),
        'f1': float(f1), 'specificity': float(spec), 'roc_auc': float(auc),
        'confusion_matrix': [[int(tn), int(fp)], [int(fn), int(tp)]]
    }

rf_res = evaluate_predictions('Random Forest Alone', y_test, test_rf_pred, test_rf_prob)
iso_res = evaluate_predictions('Isolation Forest Standalone (Benign-Trained)', y_test, test_iso_pred, test_iso_prob)
ens_res = evaluate_predictions('Hybrid Ensemble (0.6 RF + 0.4 IF, T=0.50)', y_test, test_ensemble_pred, test_ensemble_score)

print('\n' + '='*60)
print('CLASSIFICATION REPORT - HYBRID ENSEMBLE:')
print(classification_report(y_test, test_ensemble_pred, digits=4, zero_division=0))


[INFO] Running inference on untouched holdout test partition...

================== RANDOM FOREST ALONE ==================
Accuracy:    99.9533% (0.999533)
Precision:   99.9530% (0.999530)
Recall:      99.9687% (0.999687)
F1-Score:    99.9608% (0.999608)
Specificity: 99.9305%
ROC-AUC:     0.999907
Confusion Matrix: TN=25,892, FP=18, FN=12, TP=38,269

================== ISOLATION FOREST STANDALONE (BENIGN-TRAINED) ==================
Accuracy:    44.8926% (0.448926)
Precision:   63.3459% (0.633459)
Recall:      18.0220% (0.180220)
F1-Score:    28.0607% (0.280607)
Specificity: 84.5928%
ROC-AUC:     0.733849
Confusion Matrix: TN=21,918, FP=3,992, FN=31,382, TP=6,899

================== HYBRID ENSEMBLE (0.6 RF + 0.4 IF, T=0.50) ==================
Accuracy:    99.9470% (0.999470)
Precision:   99.9843% (0.999843)
Recall:      99.9269% (0.999269)
F1-Score:    99.9556% (0.999556)
Specificity: 99.9768%
ROC-AUC:     0.999872
Confusion Matrix: TN=25,904, FP=6, FN=28, TP=38,253

CLASSIFICATION REPO

### Cell 10: Calculate Curve Coordinates (ROC & PR)
Computes the Receiver Operating Characteristic (ROC) and Precision-Recall (PR) coordinates from the hybrid ensemble scores on the holdout test set.


In [10]:
# ===============================================================
# ROC & PRECISION-RECALL CURVE COORDINATES
# ===============================================================
fpr_ens, tpr_ens, _ = roc_curve(y_test, test_ensemble_score)
prec_curve, rec_curve, _ = precision_recall_curve(y_test, test_ensemble_score)

# Downsample to 100 points for smooth frontend charting
if len(fpr_ens) > 100:
    idx_r = np.linspace(0, len(fpr_ens) - 1, 100, dtype=int)
    fpr_chart = fpr_ens[idx_r].tolist()
    tpr_chart = tpr_ens[idx_r].tolist()
else:
    fpr_chart = fpr_ens.tolist()
    tpr_chart = tpr_ens.tolist()

if len(prec_curve) > 100:
    idx_p = np.linspace(0, len(prec_curve) - 1, 100, dtype=int)
    prec_chart = prec_curve[idx_p].tolist()
    rec_chart = rec_curve[idx_p].tolist()
else:
    prec_chart = prec_curve.tolist()
    rec_chart = rec_curve.tolist()

roc_data = {'fpr': fpr_chart, 'tpr': tpr_chart}
pr_data = {'precision': prec_chart, 'recall': rec_chart}
print(f'[INFO] ROC curve points: {len(fpr_chart)}, PR curve points: {len(prec_chart)}')


[INFO] ROC curve points: 100, PR curve points: 100


### Cell 11: Feature Importance Extraction
Extracts Gini feature importances from the Random Forest model across all 14 canonical features.


In [11]:
# ===============================================================
# FEATURE IMPORTANCE EXTRACTION
# ===============================================================
importances = rf.feature_importances_
feat_imp = sorted(zip(FEATURE_KEYS, importances), key=lambda x: x[1], reverse=True)
top_features = [{'feature': f, 'importance': float(imp)} for f, imp in feat_imp]

print('\n[INFO] Top Feature Importances (Random Forest):')
print('-' * 45)
print(f"{'Feature':<30s}: {'Importance'}")
print('-' * 45)
for f, imp in feat_imp:
    print(f'  {f:<30s}: {imp:.5f} ({imp*100:.2f}%)')



[INFO] Top Feature Importances (Random Forest):
---------------------------------------------
Feature                       : Importance
---------------------------------------------
  Fwd Packet Length Mean        : 0.21677 (21.68%)
  Total Length of Fwd Packets   : 0.18774 (18.77%)
  Total Fwd Packets             : 0.16266 (16.27%)
  Bwd Packet Length Mean        : 0.12048 (12.05%)
  Fwd IAT Mean                  : 0.10522 (10.52%)
  Total Backward Packets        : 0.07647 (7.65%)
  Total Length of Bwd Packets   : 0.05824 (5.82%)
  Bwd IAT Mean                  : 0.04798 (4.80%)
  Flow Duration                 : 0.01429 (1.43%)
  Flow IAT Mean                 : 0.00983 (0.98%)
  Fwd PSH Flags                 : 0.00032 (0.03%)
  Bwd PSH Flags                 : 0.00000 (0.00%)
  Fwd URG Flags                 : 0.00000 (0.00%)
  Bwd URG Flags                 : 0.00000 (0.00%)


### Cell 12: Save Models, Scaler, and Metrics
Saves the fitted `StandardScaler`, `RandomForestClassifier`, `IsolationForest`, `eval_summary.joblib`, and `metrics.json` for live inference and SOC dashboards.


In [12]:
# ===============================================================
# SAVE MODEL ARTIFACTS AND DASHBOARD METRICS
# ===============================================================
output_dir = 'models'
os.makedirs(output_dir, exist_ok=True)

summary_data = {
    'canonical_features': FEATURE_KEYS,
    'dataset_info': {
        'clean_rows': len(df_clean),
        'train_rows': len(y_train),
        'val_rows': len(y_val),
        'test_rows': len(y_test)
    },
    'isolation_forest_calibration': {
        'min_score': calib_min,
        'max_score': calib_max,
        'contamination': 0.03
    },
    'ensemble_config': {
        'alpha': 0.6,
        'beta': 0.4,
        'threshold': 0.50
    },
    'metrics_rf_alone': rf_res,
    'metrics_if_alone': iso_res,
    'metrics_ensemble': ens_res,
    'top_features': top_features,
    'roc_data': roc_data,
    'pr_data': pr_data
}

try:
    joblib.dump(scaler, os.path.join(output_dir, 'scaler.joblib'))
    joblib.dump(rf, os.path.join(output_dir, 'supervised_rf.joblib'))
    joblib.dump(iso, os.path.join(output_dir, 'unsupervised_iso.joblib'))
    joblib.dump(summary_data, os.path.join(output_dir, 'eval_summary.joblib'))
    if explainer is not None:
        joblib.dump(explainer, os.path.join(output_dir, 'shap_explainer.joblib'))
    print(f'[INFO] All model artifacts saved to {output_dir}/')
except Exception as e:
    print(f'[ERROR] Could not save joblib artifacts: {e}')

# Export metrics.json for SOC frontend dashboard
static_metrics_path = os.path.join('static', 'data', 'metrics.json')
os.makedirs(os.path.dirname(static_metrics_path), exist_ok=True)
try:
    with open(static_metrics_path, 'w') as fh:
        json.dump({
            'basic_metrics': ens_res,
            'accuracy': ens_res['accuracy'],
            'precision': ens_res['precision'],
            'recall': ens_res['recall'],
            'f1': ens_res['f1'],
            'confusion_matrix': ens_res['confusion_matrix'],
            'roc_data': roc_data,
            'pr_data': pr_data,
            'top_features': top_features,
            'rf_alone_metrics': rf_res,
            'iso_alone_metrics': iso_res,
            'model_scope': 'DDoS vs BENIGN (CIC-IDS2017 Deduplicated)',
            'canonical_features': FEATURE_KEYS
        }, fh, indent=2)
    print(f'[INFO] Metrics exported successfully to {static_metrics_path}')
except Exception as e:
    print(f'[ERROR] Could not write metrics.json: {e}')

print('\n--- [NOTEBOOK TRAINING PIPELINE COMPLETE] ---')


[INFO] All model artifacts saved to models/
[INFO] Metrics exported successfully to static\data\metrics.json

--- [NOTEBOOK TRAINING PIPELINE COMPLETE] ---
